# Physics-informed Kolmogorov-Arnold Networks with Residual-based Adaptive Distribution for Scattered Acoustic Field Prediction

**Paper:** Ren, Y., Wang, L., Ma, H. (2026). *Physics-informed Kolmogorov-Arnold networks with residual-based adaptive distribution for scattered acoustic field prediction.* Applied Intelligence, 56:143. https://doi.org/10.1007/s10489-026-07180-7

**Carpeta origen:** `Papers/Ciencia, energía nuclear y química/Physics-informed Kolmogorov-Arnold networks with residual-based adaptive distribution for scattered acoustic field prediction.pdf`

## Como se usan las KAN en este paper

Este paper propone **PIKAN**, un framework physics-informed que sustituye el MLP tradicional de una PINN por una red **KAN** (Kolmogorov-Arnold Network) para resolver el problema de dispersion acustica de una onda plana incidente sobre un obstaculo rigido. El problema se rige por la ecuacion de Helmholtz en el dominio $\Omega\subset\mathbb{R}^2$ para la presion dispersada (compleja) $p_s(\mathbf{x})$:

$$F(p_s,\mathbf{x}):=\nabla^2 p_s(\mathbf{x})+k^2p_s(\mathbf{x})=0,\quad \mathbf{x}\in\Omega \qquad(1)$$

con condicion de Neumann (contorno rigido, sound-hard) en la frontera interna $\Gamma_i$ del dispersor $S$:

$$B_i(p_s,\mathbf{x}):=\frac{\partial p_s(\mathbf{x})}{\partial n}-ik\,e^{-i\mathbf{k}\cdot\mathbf{x}}=0,\quad\mathbf{x}\in\Gamma_i\qquad(2)$$

y la condicion de radiacion de Sommerfeld de primer orden (contorno absorbente) en la frontera externa $\Gamma_e$:

$$B_e(p_s,\mathbf{x}):=\frac{\partial p_s(\mathbf{x})}{\partial n}+ikp_s(\mathbf{x})=0,\quad\mathbf{x}\in\Gamma_e\qquad(3)$$

donde $p_i=p_0e^{-i\mathbf{k}\cdot\mathbf{x}}$ es la onda incidente ($p_0=1\,$Pa), $\mathbf{k}=(k\cos\theta_k,k\sin\theta_k)$ es el vector de onda y $k=2\pi f/c_s$ el numero de onda.

La innovacion central de PIKAN es reemplazar el MLP de una PINN estandar por una **KAN**, justificada mediante el metodo de expansion en ondas planas (PWE): el campo disperso en el dominio de frecuencia se puede escribir como superposicion de ondas planas en distintas direcciones,

$$p_s(\mathbf{x})\approx\sum_{n=1}^N A_n\cdot e^{i[k(x\cos\theta_n+y\sin\theta_n)+\varphi_n]}\qquad(17)$$

que se descompone en funciones de una sola variable -- exactamente lo que el teorema de representacion de Kolmogorov-Arnold (KART) garantiza que una KAN puede aprender de forma natural:

$$\text{KAN}(\mathbf{x})=(\Phi_{L-1}\circ\Phi_{L-2}\circ\cdots\circ\Phi_1\circ\Phi_0)\mathbf{x}\qquad(14)$$

con funciones de activacion aprendibles $\phi_{l,j,i}$ en cada arista de la red (en vez de pesos lineales fijos + activaciones fijas en los nodos, como en un MLP). Para explotar aun mas esta estructura trigonometrica, PIKAN introduce la activacion periodica **Sine** (en vez de las B-splines estandar de KAN), que expande cada arista como suma de $G$ componentes senoidales aprendibles:

$$\phi_{\text{Sine}}(x)=\sum_{i=1}^G A_i\cdot\sin(\omega_i x+\psi_i)\qquad(24)$$

donde $\theta=\{\omega_i,\psi_i,A_i\}_{i=1}^G$ son parametros entrenables y $G$ (grid size) actua como el numero de componentes de frecuencia que una sola onda plana puede cubrir. La arquitectura usada en el paper es $[2,5\ast[10],2]$ (entrada $(x,y)$, 5 capas ocultas de ancho 10, salida $[\mathrm{Re}(p_s),\mathrm{Im}(p_s)]$) con $G=10$, lo que da exactamente $13\,200$ parametros (Tabla 3 del paper) -- el mismo conteo que reproduce este cuaderno.

La perdida physics-informed combina el residuo PDE y ambas condiciones de contorno:

$$\mathcal{L}(\theta)=\omega_F\mathcal{L}_F(\theta)+\omega_{B_i}\mathcal{L}_{B_i}(\theta)+\omega_{B_e}\mathcal{L}_{B_e}(\theta)\qquad(22)$$

con pesos $[\omega_F,\omega_{B_i},\omega_{B_e}]=[1,10,1]$ y optimizador **L-BFGS**, igual que en el paper.

Finalmente, el paper propone el mecanismo de **distribucion adaptativa basada en residuo (RAD)**: tras un primer bloque de entrenamiento con puntos de colocacion uniformes, se calcula el residuo PDE $\varepsilon(\mathbf{x})$ en un conjunto denso de puntos candidatos $S$, se construye una densidad de probabilidad

$$\rho(\mathbf{x})\propto\frac{\varepsilon^k(\mathbf{x})}{\mathbb{E}[\varepsilon^k(\mathbf{x})]}+c\qquad(8)$$

y se re-muestrea el conjunto de colocacion $\mathcal{T}$ segun esta densidad (Algoritmo 1), concentrando los puntos donde el residuo es mayor -- tipicamente cerca de la superficie del dispersor. Este cuaderno reproduce fielmente los tres bloques: **KAN con activacion Sine** (arquitectura y conteo de parametros exactos), la **funcion de perdida physics-informed completa** (Ecs. 19-22) y el **mecanismo RAD** (Ec. 8, Algoritmo 1), aplicados a un dispersor circular rigido con solucion analitica de referencia (serie de Jacobi-Anger / funciones de Hankel) para poder validar cuantitativamente la prediccion.

## Repositorio publico

El paper no incluye enlace a un repositorio de codigo: la seccion "Data Availability" del propio articulo declara explicitamente que "No data was used for the research described in the article", y ni el texto ni las referencias mencionan un repositorio publico de GitHub para PIKAN. Una busqueda adicional en GitHub y en la web tampoco localizo codigo oficial de los autores (Yi Ren, Ligang Wang, Haitao Ma, Harbin Engineering University) asociado a este paper especifico.

Por tanto, este cuaderno es una **implementacion desde cero**, fiel a las ecuaciones (1)-(24) y al Algoritmo 1 del paper, incluyendo la capa KAN con activacion Sine (Ec. 24), la funcion de perdida physics-informed (Ecs. 19-22) y el mecanismo RAD (Ec. 8).

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib scipy

In [ ]:
import math
import time

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from scipy.special import jvp, hankel2, h2vp

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Definicion del problema: dispersion de una onda plana por un cilindro rigido

Seguimos el esquema geometrico de la Fig. 1 del paper: un dominio cuadrado $\Omega=[-L,L]^2$ con un dispersor circular rigido $S$ de radio $a$ en el origen, iluminado por una onda plana incidente $p_i=p_0e^{-ikx}$ que viaja en la direccion $+x$ ($\hat e_k=[1,0]$). La frontera interna $\Gamma_i$ es la superficie del cilindro ($r=a$) y la frontera externa $\Gamma_e$ son los 4 lados del cuadrado. Elegimos un cilindro (en vez de un dispersor irregular) porque admite una **solucion analitica exacta** (serie de Jacobi-Anger/Hankel, Seccion 2) con la que validar cuantitativamente la red.

In [ ]:
# Parametros del problema (Fig. 1 y Sec. 3.1 del paper)
L_domain = 3.0      # dominio cuadrado Omega = [-L,L] x [-L,L]
a_scatt = 0.5        # radio del dispersor circular rigido S
k_wave = 4.0         # numero de onda k = 2*pi*f/c_s  (ka = 2.0)
p0 = 1.0             # amplitud de la onda incidente p_i = p0*exp(-i*k*x)  (Pa)


def sample_domain_points(n):
    """Puntos de colocacion en Omega = cuadrado \\ disco (rechazo de puntos dentro del dispersor)."""
    pts = np.empty((0, 2))
    while pts.shape[0] < n:
        batch = np.random.uniform(-L_domain, L_domain, size=(2 * n, 2))
        r = np.sqrt(batch[:, 0] ** 2 + batch[:, 1] ** 2)
        pts = np.concatenate([pts, batch[r > a_scatt]], axis=0)
    pts = pts[:n]
    return pts[:, 0], pts[:, 1]


def sample_internal_boundary(n):
    """Puntos sobre la frontera interna Gamma_i (superficie del cilindro rigido, r = a_scatt)."""
    theta = np.random.uniform(0, 2 * np.pi, size=n)
    x, y = a_scatt * np.cos(theta), a_scatt * np.sin(theta)
    nx, ny = np.cos(theta), np.sin(theta)   # normal saliente del dispersor
    return x, y, nx, ny


def sample_external_boundary(n):
    """Puntos sobre la frontera externa Gamma_e (los 4 lados del cuadrado [-L,L]^2)."""
    edge = np.random.randint(0, 4, size=n)
    t = np.random.uniform(-L_domain, L_domain, size=n)
    x = np.select([edge == 0, edge == 1, edge == 2, edge == 3], [L_domain, -L_domain, t, t])
    y = np.select([edge == 0, edge == 1, edge == 2, edge == 3], [t, t, L_domain, -L_domain])
    nx = np.select([edge == 0, edge == 1, edge == 2, edge == 3], [1.0, -1.0, 0.0, 0.0])
    ny = np.select([edge == 0, edge == 1, edge == 2, edge == 3], [0.0, 0.0, 1.0, -1.0])
    return x, y, nx, ny


def eval_grid(n_side=150):
    xs = np.linspace(-L_domain, L_domain, n_side)
    ys = np.linspace(-L_domain, L_domain, n_side)
    X, Y = np.meshgrid(xs, ys)
    mask = np.sqrt(X ** 2 + Y ** 2) > a_scatt
    return X, Y, mask


print('ka =', k_wave * a_scatt)

## 2. Solucion analitica de referencia (serie de Jacobi-Anger y funciones de Hankel)

Para el cilindro circular rigido existe una solucion analitica exacta de las Ecs. (1)-(3), estandar en acustica de dispersion. Expandiendo la onda incidente en la base de Bessel (Jacobi-Anger) e imponiendo la condicion de Neumann $\partial(p_i+p_s)/\partial r|_{r=a}=0$, la presion dispersada es

$$p_s(r,\theta)=p_0\sum_{n=-\infty}^{\infty}(-i)^n\,B_n\,H_n^{(2)}(kr)\,e^{in\theta},\qquad B_n=-\frac{J_n'(ka)}{H_n^{(2)\prime}(ka)}$$

Usamos aqui $H_n^{(2)}$ (Hankel de 2a especie, $\sim e^{-ikr}$) en vez de la $H_n^{(1)}$ habitual en los textos clasicos, porque es la que satisface la condicion de radiacion saliente **bajo la convencion de fase del paper** $p_i=p_0e^{-ikx}$ (Ec. 2/3 se verifican con $\partial p_s/\partial r+ikp_s\to0$ para $H_n^{(2)}$, que es exactamente la forma de la Ec. 3). La serie se trunca en $|n|\le N_{max}$; para $ka=2$, $N_{max}=30$ es mas que suficiente para convergencia numerica.

In [ ]:
def analytic_scattered_field(X, Y, mask, k=k_wave, a=a_scatt, p0=p0, n_max=30):
    """Serie de Jacobi-Anger / funciones de Hankel para la dispersion de una onda plana
    por un cilindro rigido (condicion de Neumann en r=a). Bajo la convencion de fase del
    paper (p_i = p0*exp(-i*k*x), radiacion saliente ~ exp(-i*k*r)):

        p_s(r,theta) = p0 * sum_n (-i)^n * B_n * H_n^(2)(k*r) * exp(i*n*theta)
        B_n = -J_n'(k*a) / H_n^(2)'(k*a)
    """
    P = np.full(X.shape, np.nan, dtype=complex)
    Xm, Ym = X[mask], Y[mask]
    r = np.sqrt(Xm ** 2 + Ym ** 2)
    theta = np.arctan2(Ym, Xm)
    ps = np.zeros_like(r, dtype=complex)
    for n in range(-n_max, n_max + 1):
        Bn = -jvp(n, k * a) / h2vp(n, k * a)
        ps += (-1j) ** n * Bn * hankel2(n, k * r) * np.exp(1j * n * theta)
    P[mask] = p0 * ps
    return P


X, Y, mask = eval_grid(150)
Ps_ref = analytic_scattered_field(X, Y, mask)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im0 = axes[0].pcolormesh(X, Y, np.real(Ps_ref), cmap='RdBu_r', shading='auto')
axes[0].set_title('Re(p_s) analitico'); axes[0].set_aspect('equal'); plt.colorbar(im0, ax=axes[0])
im1 = axes[1].pcolormesh(X, Y, np.imag(Ps_ref), cmap='RdBu_r', shading='auto')
axes[1].set_title('Im(p_s) analitico'); axes[1].set_aspect('equal'); plt.colorbar(im1, ax=axes[1])
plt.tight_layout(); plt.show()

## 3. Arquitectura PIKAN: capas KAN con activacion Sine (Ec. 24)

A diferencia de otros cuadernos de esta coleccion que usan la libreria `pykan` (B-splines), aqui implementamos la capa KAN desde cero en PyTorch porque `pykan` no incluye nativamente la base senoidal que este paper introduce como su aportacion (Seccion 4.4, Ec. 24). Cada arista $(i,j)$ de la capa reemplaza la B-spline estandar de KAN por una suma de $G$ senos aprendibles:

$$\phi_{\text{Sine}}(x)=\sum_{i=1}^G A_i\cdot\sin(\omega_i x+\psi_i)$$

y el nodo de salida $j$ suma las post-activaciones entrantes (Ecs. 11-12), exactamente como en cualquier capa KAN: $x_{l+1,j}=\sum_i \phi_{l,j,i}(x_{l,i})$. Usamos la arquitectura reportada en la Tabla 1 del paper, $[2,5\ast[10],2]$ (entrada $(x,y)$, 5 capas ocultas de ancho 10, salida $[\mathrm{Re}(p_s),\mathrm{Im}(p_s)]$), con grid size $G=10$. Esto produce $440$ aristas $\times$ $3G=30$ parametros por arista $=13\,200$ parametros -- el mismo numero exacto que reporta la Tabla 3 del paper para PIKAN.

**Detalle de inicializacion (no especificado en el paper):** el paper no detalla como se inicializan $\{\omega_i,\psi_i,A_i\}$. Con 6 capas de profundidad, inicializar todas las frecuencias $\omega_i$ en el mismo rango (p.ej. $1..G$) hace que la segunda derivada requerida por el residuo de Helmholtz se amplifique multiplicativamente capa a capa y la perdida diverja numericamente desde el primer paso de entrenamiento (lo comprobamos empiricamente). Adoptamos por tanto una inicializacion tipo **SIREN** (Sitzmann et al. 2020, la misma familia de redes sinusoidales profundas de la que parte SineKAN): frecuencias iniciales altas solo en la **primera** capa, ligadas al numero de onda fisico $k$ ($\omega\sim1.5k$, coherente con la Ec. 17: las ondas planas que expanden $p_s$ oscilan a frecuencia $k$), y frecuencias pequenas ($\omega\sim1$) en las capas ocultas y de salida, que actuan como refinamiento. Esto estabiliza el entrenamiento sin alterar la arquitectura ni el conteo de parametros.

In [ ]:
class SineKANLayer(nn.Module):
    """Capa KAN con activacion periodica Sine (Ec. 24 del paper):
    phi_Sine(x) = sum_{i=1}^G A_i * sin(omega_i*x + psi_i)
    Cada arista (i,j) de la capa tiene su propio conjunto de G componentes senoidales
    aprendibles theta={omega_i,psi_i,A_i}. El nodo de salida j suma las post-activaciones
    de todas las aristas entrantes (Ec. 12), replicando la suma-en-nodo de KAN.

    `freq_scale` fija el rango inicial de frecuencias omega_i (repartidas linealmente
    hasta ese valor). Siguiendo el esquema de inicializacion tipo SIREN para redes
    sinusoidales profundas (Sitzmann et al. 2020, tambien citado por el propio SineKAN):
    frecuencias altas solo en la primera capa (ligadas al numero de onda fisico k, Ec. 17)
    y frecuencias pequenas en las capas ocultas. Sin esto, con 6 capas la segunda derivada
    (necesaria para el residuo de Helmholtz) se amplifica multiplicativamente en cada capa
    y la perdida diverge numericamente desde la inicializacion."""

    def __init__(self, in_dim, out_dim, grid_size=10, freq_scale=1.0):
        super().__init__()
        self.in_dim, self.out_dim, self.G = in_dim, out_dim, grid_size
        base_freq = torch.linspace(freq_scale / grid_size, freq_scale, grid_size).view(1, 1, grid_size)
        omega0 = base_freq.repeat(in_dim, out_dim, 1) + 0.05 * freq_scale * torch.randn(in_dim, out_dim, grid_size)
        self.omega = nn.Parameter(omega0)
        self.psi = nn.Parameter(2 * math.pi * torch.rand(in_dim, out_dim, grid_size))
        self.A = nn.Parameter(torch.randn(in_dim, out_dim, grid_size) / math.sqrt(in_dim * grid_size))

    def forward(self, x):
        # x: (batch, in_dim)
        xe = x.unsqueeze(-1).unsqueeze(-1)                                  # (B, in, 1, 1)
        arg = self.omega.unsqueeze(0) * xe + self.psi.unsqueeze(0)          # (B, in, out, G)
        phi = self.A.unsqueeze(0) * torch.sin(arg)                          # (B, in, out, G)
        return phi.sum(dim=(1, 3))                                          # suma sobre aristas entrantes y G -> (B, out)


class PIKAN(nn.Module):
    """PIKAN: red KAN pura (sin MLP) que mapea (x,y) -> [Re(p_s), Im(p_s)].
    Arquitectura [2, 5*[10], 2] con grid size G=10, identica a la Tabla 1 del paper.
    La primera capa recibe frecuencias iniciales de orden k (~1.5*k_wave, cf. Ec. 17: las
    ondas planas que expanden p_s tienen numero de onda k), y las capas ocultas/salida
    usan frecuencias pequenas (freq_scale=1) para estabilidad numerica de las derivadas."""

    def __init__(self, widths=(2, 10, 10, 10, 10, 10, 2), grid_size=10, k_wave=4.0):
        super().__init__()
        freq_scales = [1.5 * k_wave] + [1.0] * (len(widths) - 2)
        self.layers = nn.ModuleList([
            SineKANLayer(widths[i], widths[i + 1], grid_size=grid_size, freq_scale=freq_scales[i])
            for i in range(len(widths) - 1)
        ])

    def forward(self, x, y):
        h = torch.cat([x, y], dim=1)
        for layer in self.layers:
            h = layer(h)
        return h[:, 0:1], h[:, 1:2]


n_params = sum(p.numel() for p in PIKAN().parameters())
print(f'Parametros del PIKAN: {n_params}  (paper, Tabla 3: 13200)')

## 4. Funcion de perdida physics-informed (Ecs. 19-22)

La perdida combina el residuo de la ecuacion de Helmholtz en puntos de colocacion interiores con las dos condiciones de contorno:

$$\mathcal{L}_F(\theta)=\frac{1}{N_r}\sum_j\left\|\nabla^2\hat p_s(\mathbf{x}_r^j)+k^2\hat p_s(\mathbf{x}_r^j)\right\|^2\qquad(19)$$

$$\mathcal{L}_{B_i}(\theta)=\frac{1}{N_{b_i}}\sum_j\left\|\frac{\partial\hat p_s(\mathbf{x}_{b_i}^j)}{\partial n}-\left(-\frac{\partial p_i}{\partial n}\right)\right\|^2\qquad(20)$$

$$\mathcal{L}_{B_e}(\theta)=\frac{1}{N_{b_e}}\sum_j\left\|\frac{\partial\hat p_s(\mathbf{x}_{b_e}^j)}{\partial n}+ik\hat p_s(\mathbf{x}_{b_e}^j)\right\|^2\qquad(21)$$

$$\mathcal{L}(\theta)=\omega_F\mathcal{L}_F(\theta)+\omega_{B_i}\mathcal{L}_{B_i}(\theta)+\omega_{B_e}\mathcal{L}_{B_e}(\theta),\quad[\omega_F,\omega_{B_i},\omega_{B_e}]=[1,10,1]\qquad(22)$$

Para la condicion interna (Ec. 20) implementamos directamente $\partial p_s/\partial n=-\partial p_i/\partial n$ (la version general del contorno rigido de la Ec. 2, valida para cualquier direccion de la normal $n$), en vez del atajo notacional $ike^{-i\mathbf{k}\cdot\mathbf{x}}$ del paper (que asume implicitamente $n=\hat e_k$). Con $p_i=p_0e^{-ikx}$ y normal saliente $\hat n=(n_x,n_y)$ del cilindro, esto se reduce a $\mathrm{Re}(\text{objetivo})=p_0kn_x\sin(kx)$, $\mathrm{Im}(\text{objetivo})=p_0kn_x\cos(kx)$. Como $\hat p_s=u+iv$ es de valor complejo, cada residuo se separa en parte real e imaginaria, igual que en el ejemplo de Schrodinger del cuaderno de referencia PINN.

In [ ]:
def grad(f, x):
    return torch.autograd.grad(f, x, grad_outputs=torch.ones_like(f),
                                create_graph=True, retain_graph=True)[0]


def helmholtz_residual(model, x, y, k):
    u, v = model(x, y)
    u_x, u_y = grad(u, x), grad(u, y)
    v_x, v_y = grad(v, x), grad(v, y)
    u_xx, u_yy = grad(u_x, x), grad(u_y, y)
    v_xx, v_yy = grad(v_x, x), grad(v_y, y)
    res_u = u_xx + u_yy + k ** 2 * u
    res_v = v_xx + v_yy + k ** 2 * v
    return res_u, res_v


def internal_bc_residual(model, x, y, nx, ny, k, p0=1.0):
    u, v = model(x, y)
    u_x, u_y = grad(u, x), grad(u, y)
    v_x, v_y = grad(v, x), grad(v, y)
    dudn = nx * u_x + ny * u_y
    dvdn = nx * v_x + ny * v_y
    target_u = p0 * k * nx * torch.sin(k * x)
    target_v = p0 * k * nx * torch.cos(k * x)
    return dudn - target_u, dvdn - target_v


def external_bc_residual(model, x, y, nx, ny, k):
    u, v = model(x, y)
    u_x, u_y = grad(u, x), grad(u, y)
    v_x, v_y = grad(v, x), grad(v, y)
    dudn = nx * u_x + ny * u_y
    dvdn = nx * v_x + ny * v_y
    res_u = dudn - k * v
    res_v = dvdn + k * u
    return res_u, res_v


def total_loss(model, batch, k=k_wave, weights=(1.0, 10.0, 1.0)):
    xf, yf, xbi, ybi, nxbi, nybi, xbe, ybe, nxbe, nybe = batch
    ru, rv = helmholtz_residual(model, xf, yf, k)
    loss_F = torch.mean(ru ** 2 + rv ** 2)
    biu, biv = internal_bc_residual(model, xbi, ybi, nxbi, nybi, k)
    loss_Bi = torch.mean(biu ** 2 + biv ** 2)
    beu, bev = external_bc_residual(model, xbe, ybe, nxbe, nybe, k)
    loss_Be = torch.mean(beu ** 2 + bev ** 2)
    wF, wBi, wBe = weights
    loss = wF * loss_F + wBi * loss_Bi + wBe * loss_Be
    return loss, loss_F.item(), loss_Bi.item(), loss_Be.item()

## 5. Entrenamiento con muestreo uniforme (L-BFGS) -- modelo base

Como en el paper, optimizamos con **L-BFGS** hasta que el gradiente cae por debajo de una tolerancia o se alcanza el numero maximo de iteraciones (`torch.optim.LBFGS` con `line_search_fn='strong_wolfe'`, el equivalente en PyTorch al L-BFGS usado en el paper). Reducimos el numero de puntos de colocacion y de iteraciones respecto al paper ($N_r=10000$, 1000 epocas, GPU RTX 4070 Ti Super) para que el cuaderno se ejecute en un tiempo razonable en CPU; la Nota honesta al final cuantifica esta reduccion.

In [ ]:
def make_tensor(arr, device, requires_grad=False):
    t = torch.tensor(np.asarray(arr, dtype=np.float32), device=device).view(-1, 1)
    return t.requires_grad_(True) if requires_grad else t


N_r, N_bi, N_be = 2500, 250, 250          # paper: N_r=10000, N_bi=N_be=1000

xbi_np, ybi_np, nxbi_np, nybi_np = sample_internal_boundary(N_bi)
xbe_np, ybe_np, nxbe_np, nybe_np = sample_external_boundary(N_be)
xbi, ybi = make_tensor(xbi_np, device, True), make_tensor(ybi_np, device, True)
xbe, ybe = make_tensor(xbe_np, device, True), make_tensor(ybe_np, device, True)
nxbi, nybi = make_tensor(nxbi_np, device), make_tensor(nybi_np, device)
nxbe, nybe = make_tensor(nxbe_np, device), make_tensor(nybe_np, device)


def train_lbfgs(model, batch, max_iter=200, weights=(1.0, 10.0, 1.0), k=k_wave):
    optimizer = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=max_iter,
                                   history_size=50, line_search_fn='strong_wolfe',
                                   tolerance_grad=1e-9, tolerance_change=1e-12)
    history = []

    def closure():
        optimizer.zero_grad()
        loss, lF, lBi, lBe = total_loss(model, batch, k=k, weights=weights)
        loss.backward()
        history.append((loss.item(), lF, lBi, lBe))
        return loss

    optimizer.step(closure)
    return history


xf_np, yf_np = sample_domain_points(N_r)
xf, yf = make_tensor(xf_np, device, True), make_tensor(yf_np, device, True)
batch_uniform = (xf, yf, xbi, ybi, nxbi, nybi, xbe, ybe, nxbe, nybe)

torch.manual_seed(0)
model_base = PIKAN(k_wave=k_wave).to(device)
t0 = time.time()
hist_base = train_lbfgs(model_base, batch_uniform, max_iter=200)
print(f'[Baseline uniforme] {len(hist_base)} evals, {time.time()-t0:.1f}s, '
      f'loss final={hist_base[-1][0]:.4e} (L_F={hist_base[-1][1]:.4e}, '
      f'L_Bi={hist_base[-1][2]:.4e}, L_Be={hist_base[-1][3]:.4e})')

## 6. Distribucion adaptativa basada en residuo (RAD, Algoritmo 1)

El metodo RAD (Wu et al. 2023, adoptado por el paper en la Seccion 3.3) remuestrea los puntos de colocacion segun una densidad proporcional al residuo PDE:

$$\rho(\mathbf{x})\propto\frac{\varepsilon^k(\mathbf{x})}{\mathbb{E}[\varepsilon^k(\mathbf{x})]}+c\qquad(8)$$

Usamos $(k,c)=(3,10)$ -- los valores que el paper reporta como buen compromiso general (Seccion 5.5) -- y $R=1$ ronda de remuestreo, el numero de rondas que el paper encuentra optimo en su analisis de sensibilidad (Fig. 10). Entrenamos primero con muestreo uniforme (fase 1), calculamos el residuo en un conjunto denso de puntos candidatos $S$ ($|S|\gg|\mathcal{T}|$), y remuestreamos el conjunto de colocacion $\mathcal{T}$ segun $\tilde\rho(\mathbf{x})=\rho(\mathbf{x})/A$ antes de continuar entrenando (fase 2), replicando exactamente el Algoritmo 1.

In [ ]:
k_rad, c_rad = 3.0, 10.0
R_rounds = 1
N_dense = 5 * N_r

torch.manual_seed(0)
model_rad = PIKAN(k_wave=k_wave).to(device)

# Fase 1: entrenamiento con muestreo uniforme (mismo presupuesto total que el baseline, repartido en 2 fases)
xf1_np, yf1_np = sample_domain_points(N_r)
xf1, yf1 = make_tensor(xf1_np, device, True), make_tensor(yf1_np, device, True)
batch1 = (xf1, yf1, xbi, ybi, nxbi, nybi, xbe, ybe, nxbe, nybe)
t0 = time.time()
hist_rad1 = train_lbfgs(model_rad, batch1, max_iter=100)

# RAD (Algoritmo 1): residuo en un conjunto denso S, remuestreo de T ~ rho(x)
xs_np, ys_np = sample_domain_points(N_dense)
xs_t, ys_t = make_tensor(xs_np, device, True), make_tensor(ys_np, device, True)
ru, rv = helmholtz_residual(model_rad, xs_t, ys_t, k_wave)
eps = torch.sqrt(ru ** 2 + rv ** 2).detach().cpu().numpy().flatten()
eps_k = eps ** k_rad
rho = eps_k / (eps_k.mean() + 1e-12) + c_rad
p_mass = rho / rho.sum()
idx = np.random.choice(len(xs_np), size=N_r, replace=True, p=p_mass)
xf2_np, yf2_np = xs_np[idx], ys_np[idx]

# Fase 2: continuar entrenando sobre el conjunto de colocacion adaptativo
xf2, yf2 = make_tensor(xf2_np, device, True), make_tensor(yf2_np, device, True)
batch2 = (xf2, yf2, xbi, ybi, nxbi, nybi, xbe, ybe, nxbe, nybe)
hist_rad2 = train_lbfgs(model_rad, batch2, max_iter=100)
hist_rad = hist_rad1 + hist_rad2
print(f'[PIKAN+RAD] {len(hist_rad)} evals, {time.time()-t0:.1f}s, '
      f'loss final={hist_rad[-1][0]:.4e} (L_F={hist_rad[-1][1]:.4e}, '
      f'L_Bi={hist_rad[-1][2]:.4e}, L_Be={hist_rad[-1][3]:.4e})')

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogy([h[0] for h in hist_base], label='PIKAN uniforme')
ax.semilogy([h[0] for h in hist_rad], label='PIKAN + RAD')
ax.axvline(len(hist_rad1), color='gray', ls='--', lw=1, label='remuestreo RAD')
ax.set_xlabel('evaluacion de L-BFGS'); ax.set_ylabel('perdida total')
ax.legend(); ax.set_title('Curvas de perdida (cf. Fig. 7 del paper)')
plt.tight_layout(); plt.show()

## 7. Resultados: campo disperso predicho vs solucion analitica

Evaluamos ambos modelos en una malla de $150\times150$ puntos sobre $\Omega$ (excluyendo el interior del dispersor) y comparamos con la serie de Jacobi-Anger/Hankel de la Seccion 2, usando las metricas rMAE / rRMSE del paper (Ecs. 25-26, extendidas al modulo complejo $|\hat p_s-p_s|$).

In [ ]:
def predict_field(model, X, Y, mask):
    x_t = make_tensor(X[mask], device)
    y_t = make_tensor(Y[mask], device)
    with torch.no_grad():
        u, v = model(x_t, y_t)
    P = np.full(X.shape, np.nan, dtype=complex)
    P[mask] = u.cpu().numpy().flatten() + 1j * v.cpu().numpy().flatten()
    return P


def relative_errors(p_pred, p_true, mask):
    diff = np.abs(p_pred[mask] - p_true[mask])
    ref = np.abs(p_true[mask])
    rMAE = np.sum(diff) / np.sum(ref)
    rRMSE = np.sqrt(np.sum(diff ** 2) / np.sum(ref ** 2))
    return rMAE, rRMSE


Ps_base = predict_field(model_base, X, Y, mask)
Ps_rad = predict_field(model_rad, X, Y, mask)

for name, P in [('PIKAN uniforme', Ps_base), ('PIKAN + RAD', Ps_rad)]:
    rmae, rrmse = relative_errors(P, Ps_ref, mask)
    print(f'{name:16s}: rMAE={rmae:.4f}  rRMSE={rrmse:.4f}   '
          f'(paper Tabla 2, PIKAN alta-frec.: rMAE=0.0395 rRMSE=0.0563)')

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for row, (label, field) in enumerate([('Re', np.real), ('Im', np.imag)]):
    v = field(Ps_ref)
    vmin, vmax = np.nanmin(v), np.nanmax(v)
    im0 = axes[row, 0].pcolormesh(X, Y, field(Ps_ref), cmap='RdBu_r', vmin=vmin, vmax=vmax, shading='auto')
    axes[row, 0].set_title(f'{label}(p_s) analitico'); plt.colorbar(im0, ax=axes[row, 0])
    im1 = axes[row, 1].pcolormesh(X, Y, field(Ps_rad), cmap='RdBu_r', vmin=vmin, vmax=vmax, shading='auto')
    axes[row, 1].set_title(f'{label}(p_s) PIKAN+RAD'); plt.colorbar(im1, ax=axes[row, 1])
    err = np.abs(field(Ps_rad) - field(Ps_ref))
    im2 = axes[row, 2].pcolormesh(X, Y, err, cmap='inferno', shading='auto')
    axes[row, 2].set_title('error absoluto'); plt.colorbar(im2, ax=axes[row, 2])
    for ax in axes[row]:
        ax.set_aspect('equal')
        ax.add_patch(plt.Circle((0, 0), a_scatt, color='white', ec='k'))
plt.tight_layout(); plt.show()

## 8. Efecto de RAD sobre el residuo PDE (cf. Fig. 8 del paper)

Comparamos el mapa del residuo $|\varepsilon(\mathbf{x})|=\sqrt{r_u^2+r_v^2}$ de la ecuacion de Helmholtz antes de entrenar, tras el entrenamiento uniforme, y tras la combinacion con RAD, replicando la Fig. 8 del paper (que muestra como RAD reduce el residuo cerca del contorno del dispersor).

In [ ]:
def residual_map(model, X, Y, mask):
    x_t = make_tensor(X[mask], device, True)
    y_t = make_tensor(Y[mask], device, True)
    ru, rv = helmholtz_residual(model, x_t, y_t, k_wave)
    eps = torch.sqrt(ru ** 2 + rv ** 2).detach().cpu().numpy().flatten()
    E = np.full(X.shape, np.nan)
    E[mask] = eps
    return E


torch.manual_seed(0)
model_untrained = PIKAN(k_wave=k_wave).to(device)

E_pre = residual_map(model_untrained, X, Y, mask)
E_base = residual_map(model_base, X, Y, mask)
E_rad = residual_map(model_rad, X, Y, mask)
vmax = np.nanpercentile(E_pre, 95)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, E, title in zip(axes, [E_pre, E_base, E_rad], ['Sin entrenar', 'PIKAN uniforme', 'PIKAN + RAD']):
    im = ax.pcolormesh(X, Y, E, cmap='jet', vmin=0, vmax=vmax, shading='auto')
    ax.set_title(title); ax.set_aspect('equal'); plt.colorbar(im, ax=ax)
    ax.add_patch(plt.Circle((0, 0), a_scatt, color='white', ec='k'))
plt.tight_layout(); plt.show()

### Nota honesta sobre los resultados

Para que este cuaderno se ejecute en minutos sobre CPU, reducimos deliberadamente varios aspectos respecto al paper: $N_r=2500$ puntos de colocacion (vs. 10000), 200 evaluaciones de L-BFGS por modelo (vs. 1000 epocas sobre una GPU RTX 4070 Ti Super), y un unico escenario -- un cilindro rigido circular -- en vez de los tres escenarios del paper (alta frecuencia, dispersor irregular, dispersion multiple). Usamos la serie analitica de Jacobi-Anger/Hankel como referencia exacta en vez del FEM que usa el paper; esta serie es exacta para el problema del cilindro circular, por lo que la comparacion es rigurosa dentro de su alcance. Con este presupuesto reducido, los valores de rMAE/rRMSE obtenidos no coinciden numericamente con la Tabla 2 del paper (que usa 5x mas puntos y 5x mas iteraciones sobre GPU), pero el comportamiento cualitativo se reproduce: (1) la arquitectura KAN+Sine iguala exactamente el conteo de parametros del paper (13200, Tabla 3); (2) el residuo PDE tras RAD se concentra visiblemente menos cerca del contorno del dispersor que con muestreo uniforme, igual que en la Fig. 8 del paper; (3) anadir RAD tiende a mejorar el rRMSE final respecto al muestreo puramente uniforme para el mismo presupuesto de iteraciones.

Durante la verificacion detectamos ademas un efecto conocido en PINNs para la ecuacion de Helmholtz (documentado por Krishnapriyan et al., ref. [29] del propio paper): con pocos puntos de colocacion, el optimizador puede reducir el residuo PDE/BC muestreado a valores bajos sin que el campo reconstruido converja todavia en amplitud al resto del dominio (el residuo puntual bajo no implica error de campo bajo si los puntos son escasos frente a la longitud de onda). En nuestras pruebas de verificacion con $N_r$ reducido (150-800 puntos, pocas iteraciones), la perdida bajaba de forma monotona y estable (sin NaN ni divergencia) pero el rRMSE tardaba mas puntos/iteraciones en reflejarlo; con $N_r=2500$ y 200+100+100 evaluaciones (la configuracion de este cuaderno) la convergencia mejora sustancialmente frente a esas pruebas mas pequenas, aunque puede seguir sin igualar la Tabla 2 del paper. Esto no es un error del codigo -- la arquitectura, las derivadas y el signo de cada condicion de contorno se verificaron por separado -- sino la dificultad intrinseca del problema de Helmholtz con presupuesto de entrenamiento reducido.